## MI PRIMER BOT DE TELEGRAM CON PYTHON
## imports de librerias

In [ ]:
import telebot
from dotenv import load_dotenv
import os
import time
import random
import json
import html
from telebot.types import InlineKeyboardMarkup
from telebot.types import InlineKeyboardButton
from telebot.types import ForceReply

# Configurando TMDB y Instanciando al bot :)

In [6]:
import os
import requests
import telebot
import pathlib
from dotenv import load_dotenv

# 1. Le decimos a Python que se pare exactamente en la carpeta de tu bot
os.chdir(pathlib.Path().resolve())

# 2. Ahora load_dotenv() y os.getenv() van a funcionar sin dar vueltas
load_dotenv(override=True)
TELEGRAM_TOKEN = os.getenv("TELEGRAM_TOKEN")
TMDB_TOKEN = os.getenv("PELICULAS_TOKEN")

TMDB_BASE_URL = "https://api.themoviedb.org/3"

# Configurando OMDB

In [18]:
API_KEY= os.getenv("API_KEY")

def buscar_en_omdb(nombre_pelicula):
    url = f"https://omdbapi.com{API_KEY}&t={nombre_pelicula}"
    respuesta = requests.get(url)
    datos = respuesta.json()
    return datos

# Instancio al bot

In [8]:
tkn = os.getenv("TELEGRAM_TOKEN")
bot = telebot.TeleBot(tkn)

# Lista de peliculas, anda con ganas de:

In [10]:
comicas = [
    "The Hangover", "Superbad", "Step Brothers", "Anchorman: The Legend of Ron Burgundy",
    "Dumb and Dumber", "Bridesmaids", "Airplane!", "Ghostbusters", "Groundhog Day",
    "The Big Lebowski", "Zoolander", "Mean Girls", "Ace Ventura: Pet Detective", "The Mask",
    "Home Alone", "Elf", "Napoleon Dynamite", "Borat", "Wedding Crashers", "21 Jump Street",
    "Tropic Thunder", "Shaun of the Dead", "Hot Fuzz", "Monty Python and the Holy Grail",
    "Blades of Glory", "Old School", "Meet the Parents", "Rush Hour", "Coming to America",
    "Bill & Ted's Excellent Adventure", "Clueless", "The Grand Budapest Hotel", "Jojo Rabbit",
    "Superbad", "Booksmart",
]

accion = [
    "Die Hard", "Mad Max: Fury Road", "John Wick", "The Dark Knight", "Gladiator",
    "Terminator 2: Judgment Day", "Speed", "Raiders of the Lost Ark",
    "Mission: Impossible - Fallout", "Casino Royale", "Skyfall", "The Matrix", "Lethal Weapon",
    "Heat", "Top Gun: Maverick", "Kill Bill: Vol. 1", "Predator", "Aliens", "Bad Boys",
    "Fast Five", "The Bourne Ultimatum", "Inception", "Avengers: Endgame", "Black Panther",
    "Braveheart", "Léon: The Professional", "The Raid", "Crouching Tiger, Hidden Dragon",
    "Edge of Tomorrow", "Baby Driver", "Mad Max 2", "逆襲のシャア".replace("逆襲のシャア", "Point Break"),
    "Man on Fire", "Taken", "The Equalizer",
]

motivacion = [
    "Rocky", "The Pursuit of Happyness", "Forrest Gump", "The Shawshank Redemption",
    "Good Will Hunting", "Remember the Titans", "Invictus", "Rudy", "Hidden Figures",
    "The Blind Side", "Coach Carter", "Whiplash", "Cool Runnings", "Moneyball",
    "Dead Poets Society", "A Beautiful Mind", "The Social Network", "Erin Brockovich",
    "Miracle", "Seabiscuit", "Chariots of Fire", "Creed", "The Karate Kid", "Cinderella Man",
    "Field of Dreams", "October Sky", "Soul", "The Theory of Everything", "Freedom Writers",
    "127 Hours", "Warrior", "Ford v Ferrari", "Rocky Balboa", "Jerry Maguire", "Any Given Sunday",
]

relajado = [
    "Spirited Away", "My Neighbor Totoro", "Amélie", "Paddington 2", "The Princess Bride",
    "Up", "Finding Nemo", "Toy Story", "Coco", "Ratatouille", "WALL·E", "Big Fish",
    "Little Miss Sunshine", "About Time", "Notting Hill", "Love Actually", "Before Sunrise",
    "La La Land", "Chef", "Julie & Julia", "The Secret Life of Walter Mitty",
    "Kiki's Delivery Service", "Howl's Moving Castle", "Mamma Mia!", "Lost in Translation",
    "Little Women", "Pride & Prejudice", "The Intern", "Shrek", "Moana", "Inside Out",
    "Cinema Paradiso", "Life Is Beautiful", "Chocolat", "Sing Street",
]

# Funciones/comandos

In [ ]:
@bot.message_handler(commands=["start"])
def cmd_start(message):
    bot.reply_to(message, "Hola, Soy un bot que te puedo ayudar con..."),

@bot.message_handler(commands=["saludar"])
def cmd_saludar(message):
    bot.reply_to(message, "Hola soy el bot inicial, ¿cómo estás?")

@bot.message_handler(commands=["fin"])
def cmd_break(message):
    bot.reply_to(message, "Hasta luego!!")

# Haciendo la botonera
LISTAS = {
    "comicas": comicas,
    "accion": accion,
    "motivacionales": motivacion,
    "relajado": relajado,
}

ultima_pelicula = {}  # guarda la última recomendada de cada chat


@bot.callback_query_handler(func=lambda call: call.data.startswith("humor_"))
def procesar_emocion(call):
    humor = call.data.replace("humor_", "")

    if humor == "divertido":
        peliculas = comicas
    elif humor == "accion":
        peliculas = accion
    elif humor == "triste":
        peliculas = motivacion
    elif humor == "relajado":
        peliculas = relajado
    else:
        return "No se encontró la pelicula"

    numero = random.randint(0, len(peliculas) - 1)
    pelicula = peliculas[numero]

    markup = InlineKeyboardMarkup() #Crea una botonera vacía.
    markup.add(InlineKeyboardButton("🔍 Ver detalles", callback_data="detalles_" + pelicula)) #Crea un botón y lo mete en la botonera. El usuario ve solo "🔍 Ver detalles". callback_data es el dato escondido que te llega al tocarlo. "detalles_" + pelicula pega los dos textos, por ejemplo "detalles_Rocky".

    bot.answer_callback_query(call.id) #identifica el clic, no a la persona ni un número que escribió. Cada vez que se toca un botón, Telegram genera un id nuevo para ese clic. 
    #bot.answer_callback_query(call.id) le dice a Telegram "recibí ese clic". Solo sirve para sacarle el reloj de carga al botón (y, si le pasás text=, mostrar un aviso chiquito arriba). No manda ningún mensaje al chat.
 
    bot.send_message(call.message.chat.id, f"🎬 Te recomiendo: <tg-spoiler>{html.escape(pelicula)}</tg-spoiler>", reply_markup=markup) # El reply_markup es lo que pega la botonera al mensaje, manda los botones junto con el mensaje, en el mismo envío.. Sin ese parámetro, el mensaje se manda sin botones
#Manda el mensaje al chat donde tocaron el botón, con el título en el texto y la botonera pegada debajo.

@bot.callback_query_handler(func=lambda call: call.data.startswith("detalles_")) #Le dice a telebot que esta función atienda solo los clics cuyo callback_data empieza con detalles_.
def procesar_detalles(call):  #Define la función. call es el clic que recibe.
    pelicula = call.data.replace("detalles_", "") #Borra el prefijo y recupera el título. "detalles_Rocky" queda "Rocky".
    bot.answer_callback_query(call.id) #Confirma que se recibió el clik

    respuesta = requests.get(
        "https://www.omdbapi.com/",
        params={"apikey": API_KEY, "t": pelicula}, #Es el nuevo endpoint
    )
    datos = respuesta.json()

    if datos["Response"] == "True":
        mensaje = (
            f"Título: {datos['Title']}\n"
            f"Año: {datos['Year']}\n"
            f"Género: {datos['Genre']}\n"
            f"Director: {datos['Director']}\n"
            f"Actores: {datos['Actors']}\n"
            f"Duración: {datos['Runtime']}\n"
            f"IMDb: {datos['imdbRating']}\n\n"
            f"Sinopsis: {datos['Plot']}"
        )
        bot.send_message(call.message.chat.id, mensaje) #Manda los mensajes
    else:
        bot.send_message(call.message.chat.id, "No se encontró la película.") #Dice que no se encontró
        
# call es el clic en el botón.
# call.message es el mensaje al que estaba pegado ese botón (el de "🎬 Te recomiendo: Rocky").
# call.message.chat es el chat donde está ese mensaje.
# call.message.chat.id es el número que identifica a ese chat.

# Setear comandos

In [21]:
bot.set_my_commands([
    telebot.types.BotCommand("/start","Saludo inicial")
])

True

## Subir el bot

In [ ]:
bot.set_my_commands([
    telebot.types.BotCommand("/start","Bienvenida al bot"),
    telebot.types.BotCommand("/saludar","Saludo"),
    telebot.types.BotCommand("/fin","Cortar charla"),
])

In [ ]:
if __name__ == "__main__":

    print ("Iniciando el bot")

    bot.infinity_polling()

    print("Fin")